# Module 03: Decoder / Encoder / Multiplexer Simulation
**AE2.021 · USTH**

This notebook simulates the truth table of the three most common MSI circuits in this module: a 3-to-8
decoder, a priority encoder, and 4-to-1 / 8-to-1 multiplexers: and verifies the n = log₂(N) formula.

## 1. The 3-to-8 decoder
🎯 **What question does this method answer?** With 3 address lines, how to determine exactly which output line is activated (=1) for EACH address combination, without mixing up the bit order.

In [1]:
import itertools

def decoder_3to8(a2, a1, a0):
    addr = (a2 << 2) | (a1 << 1) | a0
    outputs = [0]*8
    outputs[addr] = 1
    return outputs

print("A2 A1 A0 | Y0 Y1 Y2 Y3 Y4 Y5 Y6 Y7")
for a2, a1, a0 in itertools.product([0,1], repeat=3):
    outs = decoder_3to8(a2, a1, a0)
    print(f" {a2}  {a1}  {a0}  | " + " ".join(str(o) for o in outs))

A2 A1 A0 | Y0 Y1 Y2 Y3 Y4 Y5 Y6 Y7
 0  0  0  | 1 0 0 0 0 0 0 0
 0  0  1  | 0 1 0 0 0 0 0 0
 0  1  0  | 0 0 1 0 0 0 0 0
 0  1  1  | 0 0 0 1 0 0 0 0
 1  0  0  | 0 0 0 0 1 0 0 0
 1  0  1  | 0 0 0 0 0 1 0 0
 1  1  0  | 0 0 0 0 0 0 1 0
 1  1  1  | 0 0 0 0 0 0 0 1


#### 📤 Real output
As expected: in every row EXACTLY ONE output equals 1 (the output whose index equals the binary value of the address A2A1A0) and the other 7 are 0: the core property of a decoder.

## 2. The multiplexer and the formula n = log₂(N)
🎯 **What question does this method answer?** With exactly N data channels, how many select lines are needed at minimum: and confirm that n=log₂(N) matches the questions on the website (a 4-to-1 mux needs 2 select lines, an 8-to-1 needs 3).

In [2]:
import math

def mux(data, select_bits):
    idx = int("".join(str(b) for b in select_bits), 2)
    return data[idx]

# 4-to-1 mux, needs 2 select lines
data4 = [0,1,0,1]
for s1, s0 in itertools.product([0,1], repeat=2):
    print(f"S1={s1} S0={s0} -> Y = data[{int(f'{s1}{s0}',2)}] = {mux(data4,[s1,s0])}")

for N in (2,4,8,16,32):
    n = math.log2(N)
    print(f"N={N:2d} channels -> need n = log2({N}) = {n:.0f} select lines")

S1=0 S0=0 -> Y = data[0] = 0
S1=0 S0=1 -> Y = data[1] = 1
S1=1 S0=0 -> Y = data[2] = 0
S1=1 S0=1 -> Y = data[3] = 1
N= 2 channels -> need n = log2(2) = 1 select lines
N= 4 channels -> need n = log2(4) = 2 select lines
N= 8 channels -> need n = log2(8) = 3 select lines
N=16 channels -> need n = log2(16) = 4 select lines
N=32 channels -> need n = log2(32) = 5 select lines


#### 📤 Real output
The table confirms: N=4 → n=2 select lines, N=8 → n=3 select lines: matching the original question "A four-to-one
multiplexer has two select inputs" (Tooley Ch.9 Q5) and the generated question on the 8-to-1 mux.

## 3. A simple priority encoder (8 inputs)
🎯 **What question does this method answer?** When SEVERAL inputs are active at once, a priority encoder encodes the input with the HIGHEST index: unlike a plain encoder (correct only when exactly one input is active).

In [3]:
def priority_encoder_8to3(inputs):
    for i in range(7, -1, -1):
        if inputs[i] == 1:
            return format(i, '03b')
    return None

tests = [
    [0,0,0,0,0,0,1,0],  # only input 6 active
    [0,0,1,0,0,1,0,1],  # several inputs active: 2, 5, 7 -> priority goes to 7
]
for t in tests:
    print(f"input={t} -> encoded (highest priority) = {priority_encoder_8to3(t)}")

input=[0, 0, 0, 0, 0, 0, 1, 0] -> encoded (highest priority) = 110
input=[0, 0, 1, 0, 0, 1, 0, 1] -> encoded (highest priority) = 111


#### 📤 Real output
When inputs 2, 5 and 7 are all active, the priority encoder correctly picks input 7 (highest index) and encodes it as
`111`, illustrating why an encoder needs extra "priority" logic compared with simple encoding.

---
## 4. Floyd problem bank: Chapter 6 "Problems" (Functions of Combinational Logic)

Chapter 6 has 55 problems but **most rely on specific circuit figures** (Figure 6-69 to 6-9x:
multi-stage adders, comparator circuits, IC pin diagrams...) that would need image extraction to
solve accurately. This notebook solves the **4 problems that need no figure** (1, 2, 3: pure adder
logic with A, B, Cin given as text; 9: using the exact 8 bit-sequences given in the problem, no
substitute data).

In [4]:
# -*- coding: utf-8 -*-
"""Auto-solve the text-only (no-figure) problems from Floyd Ch.6 (adders) and Ch.11 (memory)."""

print("="*70, "\nFloyd Ch.6 - Section 6-1 Half and Full Adders (Problems 1-3)\n" + "="*70)
def full_adder(a,b,cin):
    s = a ^ b ^ cin
    cout = (a & b) | (cin & (a ^ b))
    return s, cout

print("1) Full adder, find Sum(S) and Carry-out(Cout):")
for a,b,cin in [(0,1,0),(1,0,1),(0,0,0)]:
    s,c = full_adder(a,b,cin)
    print(f"   A={a},B={b},Cin={cin} -> Sum={s}, Cout={c}")

print("2) Half-adder inputs needed to give the desired Sum/Cout (Sum=A xor B, Cout=A and B):")
targets = [(0,0),(1,0),(0,1)]
for s_target, c_target in targets:
    found = [(a,b) for a in (0,1) for b in (0,1) if (a^b)==s_target and (a&b)==c_target]
    print(f"   Want Sum={s_target},Cout={c_target} -> (A,B) can be: {found}")

print("3) Full adder with 4 sets of values:")
for a,b,cin in [(1,0,0),(0,0,1),(0,1,1),(1,1,1)]:
    s,c = full_adder(a,b,cin)
    print(f"   A={a},B={b},Cin={cin} -> Sum={s}, Cout={c}")

print("\n" + "="*70, "\nFloyd Ch.6 - Section 6-2 Problem 9 (REAL data from the book, read in sequence over 4 clock cycles)\n" + "="*70)
print("   Problem: A1=1010, A2=1100, A3=0101, A4=1101, B1=1001, B2=1011, B3=0000, B4=0001")
print("   (\"right-most bit first\" -> the RIGHT-most bit of each string is the value at cycle t=1)")
raw = {"A1":"1010","A2":"1100","A3":"0101","A4":"1101","B1":"1001","B2":"1011","B3":"0000","B4":"0001"}
# seq[line][t] with t=0..3 (t=0 is the first cycle = the right-most bit)
seq = {k: list(reversed(v)) for k,v in raw.items()}
print("   After reordering (cycles 1->4 read from left to right):")
for k,v in seq.items(): print(f"     {k}: {''.join(v)}  (chu ky1={v[0]}, chu ky2={v[1]}, chu ky3={v[2]}, chu ky4={v[3]})")
print("\n   At each cycle t, A = A4A3A2A1 (MSB..LSB), B = B4B3B2B1 (MSB..LSB), compute Sum = A+B:")
sums = []
for t in range(4):
    Aval = int(seq["A4"][t]+seq["A3"][t]+seq["A2"][t]+seq["A1"][t], 2)
    Bval = int(seq["B4"][t]+seq["B3"][t]+seq["B2"][t]+seq["B1"][t], 2)
    s = Aval + Bval
    sums.append(format(s, '05b'))  # 5 bit: Cout Sigma4 Sigma3 Sigma2 Sigma1
    print(f"     Cycle {t+1}: A={Aval:2d} ({format(Aval,'04b')}), B={Bval:2d} ({format(Bval,'04b')}) -> Sum={s:2d} = {format(s,'05b')} (Cout Sig4 Sig3 Sig2 Sig1)")
print("\n   => Bit string on EACH sum line (read over 4 cycles, cycle 1 first):")
for i, label in enumerate(["Cout","Sigma4","Sigma3","Sigma2","Sigma1"]):
    bits = "".join(s[i] for s in sums)
    print(f"     {label}: {bits}")

print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Semiconductor Memory Basics (Problem 4)\n" + "="*70)
print("4) Memory address (0-256) corresponding to each hex number:")
for h in ["0C","5E","DF"]:
    print(f"   {h}16 = {int(h,16)} (decimal)")

print("\n" + "="*70, "\nFloyd Ch.11 - Problem 2 (concept, no calculation): relation between bit/byte/nibble/word\n" + "="*70)
print("   1 nibble = 4 bit. 1 byte = 8 bit = 2 nibble. 1 word = a multiple of a byte (usually 16/32/64-bit = 2/4/8 bytes),")
print("   the word width depends on the specific CPU architecture.")

print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Problem 5 (static RAM with 4 rows, given conditions)\n" + "="*70)
rows = [0,0,0,0]
conditions = [(0,1,1),(1,0,1),(2,1,0),(3,0,1)]  # (row_index, row_select, data_in_bit)
print("   Initially all 4 rows = 0. Apply the conditions (write only when Row select=1):")
for row_idx, sel, data in conditions:
    if sel == 1:
        rows[row_idx] = data
        print(f"   Row {row_idx} selected (select=1), writes Data in={data} -> Row {row_idx} = {data}")
    else:
        print(f"   Row {row_idx} NOT selected (select=0) -> unchanged = {rows[row_idx]}")
print("   Final contents of the 4 rows:", rows)


Floyd Ch.6 - Section 6-1 Half and Full Adders (Problems 1-3)
1) Full adder, find Sum(S) and Carry-out(Cout):
   A=0,B=1,Cin=0 -> Sum=1, Cout=0
   A=1,B=0,Cin=1 -> Sum=0, Cout=1
   A=0,B=0,Cin=0 -> Sum=0, Cout=0
2) Half-adder inputs needed to give the desired Sum/Cout (Sum=A xor B, Cout=A and B):
   Want Sum=0,Cout=0 -> (A,B) can be: [(0, 0)]
   Want Sum=1,Cout=0 -> (A,B) can be: [(0, 1), (1, 0)]
   Want Sum=0,Cout=1 -> (A,B) can be: [(1, 1)]
3) Full adder with 4 sets of values:
   A=1,B=0,Cin=0 -> Sum=1, Cout=0
   A=0,B=0,Cin=1 -> Sum=1, Cout=0
   A=0,B=1,Cin=1 -> Sum=0, Cout=1
   A=1,B=1,Cin=1 -> Sum=1, Cout=1

Floyd Ch.6 - Section 6-2 Problem 9 (REAL data from the book, read in sequence over 4 clock cycles)
   Problem: A1=1010, A2=1100, A3=0101, A4=1101, B1=1001, B2=1011, B3=0000, B4=0001
   ("right-most bit first" -> the RIGHT-most bit of each string is the value at cycle t=1)
   After reordering (cycles 1->4 read from left to right):
     A1: 0101  (chu ky1=0, chu ky2=1, chu ky3=0,